# 06 - Pandas: Time Series and Window Functions

**Prerequisite:** notebooks 04-05.
**This notebook:** datetimes, the `DatetimeIndex`, `resample`, rolling windows, and the full mapping to SQL `OVER (...)` clauses.

Time series is where Pandas genuinely beats SQL for exploratory work: resampling and rolling windows are one method call each.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 25)

DATA = next(p for p in [Path("../data"), Path("data"), Path("study/data")] if p.exists())
orders    = pd.read_csv(DATA / "orders.csv", parse_dates=["order_ts"])
customers = pd.read_csv(DATA / "customers.csv", parse_dates=["signup_date"])
daily     = pd.read_csv(DATA / "daily_metrics.csv", parse_dates=["date"])
print(daily.head(3).to_string(index=False))

## 1. Parsing and the `.dt` accessor

`datetime64[ns]` is a 64-bit integer count of nanoseconds since 1970-01-01. Java equivalent: `Instant`. Add a timezone and it becomes closer to `ZonedDateTime`.

The `.dt` accessor exposes date parts on a whole column at once - the vectorized version of calling getters on each `LocalDateTime`.

In [ ]:
ts = orders["order_ts"]
parts = pd.DataFrame({
    "ts": ts,
    "year": ts.dt.year,
    "month": ts.dt.month,
    "day": ts.dt.day,
    "hour": ts.dt.hour,
    "dayofweek": ts.dt.dayofweek,          # Monday = 0
    "day_name": ts.dt.day_name(),
    "quarter": ts.dt.quarter,
    "is_month_end": ts.dt.is_month_end,
    "date_only": ts.dt.date,
    "iso_week": ts.dt.isocalendar().week,
})
print(parts.head(5).to_string(index=False))

In [ ]:
# Parsing awkward input
print(pd.to_datetime(["2026-09-14", "14/09/2026", "Sep 14, 2026"], format="mixed", dayfirst=True))
print()
print("explicit format is much faster on big columns:")
print(pd.to_datetime(["14-09-2026"], format="%d-%m-%Y"))
print()
print("bad values -> NaT instead of an exception:")
print(pd.to_datetime(["2026-09-14", "not a date"], errors="coerce"))
print()
print("from a unix epoch:", pd.to_datetime([1757836800], unit="s"))

## 2. `DatetimeIndex` - put time in the index

Once a datetime column is the index you get partial-string slicing, resampling, and time-aware joins.

In [ ]:
d = daily.set_index("date")
print(d.head(3))
print()
print("type of index:", type(d.index).__name__)
print()
print("partial string slicing - a whole month, no explicit range:")
print(d.loc["2025-03"].head(3))
print()
print("a date range (label slicing, END INCLUSIVE):")
print(d.loc["2025-03-01":"2025-03-04"])

In [ ]:
print("last 5 rows by time:", d.last_valid_index())
print()
print("between_time / at_time work on intraday data; for daily data use plain slicing")
print()
print("truncate:", d.truncate(before='2026-08-25').shape)

### `date_range` and frequency aliases

In [ ]:
print("daily      :", pd.date_range("2026-01-01", periods=4, freq="D").tolist())
print("business   :", pd.date_range("2026-01-01", periods=4, freq="B").tolist())
print("weekly Sun :", pd.date_range("2026-01-01", periods=3, freq="W").tolist())
print("month start:", pd.date_range("2026-01-01", periods=3, freq="MS").tolist())
print("month end  :", pd.date_range("2026-01-01", periods=3, freq="ME").tolist())
print("quarter end:", pd.date_range("2026-01-01", periods=3, freq="QE").tolist())
print("hourly     :", pd.date_range("2026-01-01", periods=3, freq="h").tolist())
print("15 minutes :", pd.date_range("2026-01-01", periods=3, freq="15min").tolist())

> **Version note.** Pandas renamed the frequency aliases: `M` -> `ME`, `Q` -> `QE`, `Y`/`A` -> `YE`, `H` -> `h`, `T` -> `min`, `S` -> `s`. The old single-letter forms were removed in Pandas 3.x. Tutorials and Stack Overflow answers written before 2024 will use the old ones and raise `ValueError: Invalid frequency`.

## 3. `resample` - GROUP BY on a time bucket

`resample("ME")` is `GROUP BY DATE_TRUNC('month', ts)`. It requires a DatetimeIndex (or `on=`).

In [ ]:
monthly = d.resample("ME").agg(
    visits=("visits", "sum"),
    conversions=("conversions", "sum"),
    days=("visits", "size"),
)
monthly["conv_rate_pct"] = (monthly["conversions"] / monthly["visits"] * 100).round(2)
print(monthly.head(6))

In [ ]:
# Resample on a column without setting the index, using on=
by_week = orders.resample("W", on="order_ts").agg(
    n_orders=("order_id", "count"),
    revenue=("amount", "sum"),
).round(0)
print(by_week.head(5))

In [ ]:
# Upsampling: go finer, then decide how to fill the gaps
small = d[["visits"]].head(3)
print("original (daily):")
print(small)
print()
print("resampled to 12h with forward fill:")
print(small.resample("12h").ffill())
print()
print("with interpolation:")
print(small.resample("12h").interpolate().round(1))

| Time operation | SQL | Pandas |
|---|---|---|
| bucket by month | `GROUP BY DATE_TRUNC('month', ts)` | `resample("ME")` |
| bucket by week | `DATE_TRUNC('week', ts)` | `resample("W")` |
| calendar label only | `DATE_TRUNC` | `df["ts"].dt.to_period("M")` |
| fill gaps in a series | a calendar table + `LEFT JOIN` | `reindex(date_range(...))` or `asfreq()` |

In [ ]:
# asfreq exposes missing dates explicitly - the calendar-table trick, built in
sparse = d[["visits"]].iloc[[0, 3, 7]]
print("sparse:\n", sparse)
print()
print("asfreq('D') inserts the missing calendar days as NaN:\n", sparse.asfreq("D"))

## 4. Rolling windows = `OVER (ORDER BY ... ROWS BETWEEN ...)`

In [ ]:
w = d[["visits"]].copy()
w["visits"] = w["visits"].ffill()          # fill the injected gaps first

w["ma_7"]   = w["visits"].rolling(7).mean().round(1)             # 7-row moving average
w["ma_7_min1"] = w["visits"].rolling(7, min_periods=1).mean().round(1)
w["max_7"]  = w["visits"].rolling(7).max()
w["std_7"]  = w["visits"].rolling(7).std().round(2)
w["centered"] = w["visits"].rolling(7, center=True).mean().round(1)

print(w.head(9))

> `rolling(7)` needs 7 observations before it emits a value, so the first 6 rows are NaN. `min_periods=1` starts immediately with a partial window. `center=True` puts the label in the middle of the window instead of at the end (do not use it for anything predictive - it looks into the future).

In [ ]:
# Time-based windows: '7D' means 7 calendar days, not 7 rows.
# This is the right choice when rows are irregularly spaced.
w["ma_7D"] = w["visits"].rolling("7D").mean().round(1)
print(w[["visits", "ma_7", "ma_7D"]].head(9))

In [ ]:
# expanding = running/cumulative window, from the start to the current row
w["cum_visits"] = w["visits"].expanding().sum()
w["running_max"] = w["visits"].expanding().max()
w["cum_mean"] = w["visits"].expanding().mean().round(1)
print(w[["visits", "cum_visits", "running_max", "cum_mean"]].head(6))

In [ ]:
# ewm = exponentially weighted - recent points matter more. No SQL equivalent.
w["ewm_10"] = w["visits"].ewm(span=10).mean().round(1)
print(w[["visits", "ma_7", "ewm_10"]].head(8))

## 5. `shift`, `diff`, `pct_change` - LAG and LEAD

In [ ]:
v = d[["visits"]].ffill().head(8).copy()
v["prev_day"]   = v["visits"].shift(1)         # LAG(visits, 1)
v["next_day"]   = v["visits"].shift(-1)        # LEAD(visits, 1)
v["day_change"] = v["visits"].diff()           # visits - LAG(visits)
v["pct_change"] = (v["visits"].pct_change() * 100).round(2)
v["week_ago"]   = v["visits"].shift(7)         # LAG(visits, 7)
print(v)

In [ ]:
# shift with a time offset shifts by TIME, not by row count - safe with gaps
print(d[["visits"]].head(4).assign(shifted=lambda x: x["visits"].shift(freq="1D")).head(4))

### Per-group time series

Everything above combines with `groupby`. This is `OVER (PARTITION BY k ORDER BY ts)`.

In [ ]:
ch = (orders
      .set_index("order_ts")
      .groupby("channel")
      .resample("ME")["amount"].sum()
      .rename("revenue")
      .round(0)
      .reset_index())

ch["prev_month"]  = ch.groupby("channel")["revenue"].shift(1)
ch["mom_pct"]     = (ch["revenue"] / ch["prev_month"] - 1).mul(100).round(1)
ch["rolling_3m"]  = ch.groupby("channel")["revenue"].transform(lambda s: s.rolling(3).mean()).round(0)
ch["cumulative"]  = ch.groupby("channel")["revenue"].cumsum().round(0)

print(ch.head(8).to_string(index=False))

SQL equivalent of the last block:

```sql
SELECT channel, month, revenue,
       LAG(revenue) OVER (PARTITION BY channel ORDER BY month)                     AS prev_month,
       AVG(revenue) OVER (PARTITION BY channel ORDER BY month
                          ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                AS rolling_3m,
       SUM(revenue) OVER (PARTITION BY channel ORDER BY month)                     AS cumulative
FROM monthly_revenue;
```

## 6. Periods vs Timestamps

A `Timestamp` is an instant. A `Period` is a **span** - "March 2025" rather than "2025-03-01 00:00:00". Periods are the right type for monthly reporting labels.

In [ ]:
p = pd.Period("2026-03", freq="M")
print("period       :", p)
print("start / end  :", p.start_time, "->", p.end_time)
print("arithmetic   :", p + 1)
print()
print("on a column:")
print(orders["order_ts"].dt.to_period("M").head(3).tolist())
print()
print("group by month label:")
print(orders.groupby(orders["order_ts"].dt.to_period("M"))["amount"].sum().head(4).round(0))

## 7. Timezones

A `datetime64[ns]` column is **naive** - no zone. `tz_localize` attaches a zone to naive data (it does not shift the clock); `tz_convert` moves an aware timestamp to another zone (it does shift the displayed clock).

In [ ]:
naive = pd.Series(pd.date_range("2026-09-14 09:00", periods=3, freq="h"))
print("naive       :", naive.dt.tz, "\n", naive.tolist())
print()
aware = naive.dt.tz_localize("Asia/Kolkata")
print("localized   :", aware.dt.tz)
print(aware.tolist())
print()
print("converted to UTC (same instants, different wall clock):")
print(aware.dt.tz_convert("UTC").tolist())

> **Rule that saves incidents:** store and compute in UTC, convert to a local zone only for display. Same rule as `Instant` vs `ZonedDateTime` in Java. Mixing naive and aware timestamps raises `TypeError`, which is the behaviour you want.

## 8. `merge_asof` - join on nearest time

A time-aware join with no SQL equivalent short of a correlated subquery. For each row on the left, take the most recent row on the right at or before that time. Standard for joining trades to quotes, or events to the configuration in force at that moment.

In [ ]:
events = pd.DataFrame({
    "ts": pd.to_datetime(["2026-01-01 10:00", "2026-01-01 10:07", "2026-01-01 10:15"]),
    "event": ["click", "purchase", "click"],
})
prices = pd.DataFrame({
    "ts": pd.to_datetime(["2026-01-01 09:55", "2026-01-01 10:05", "2026-01-01 10:12"]),
    "price": [100.0, 102.5, 101.0],
})

joined = pd.merge_asof(events.sort_values("ts"), prices.sort_values("ts"), on="ts", direction="backward")
print(joined.to_string(index=False))
print()
print("with a tolerance - no match beyond 4 minutes:")
print(pd.merge_asof(events, prices, on="ts", direction="backward",
                    tolerance=pd.Timedelta("4min")).to_string(index=False))

## 9. Business days and offsets

In [ ]:
from pandas.tseries.offsets import BDay, MonthEnd, DateOffset

d0 = pd.Timestamp("2026-09-14")     # a Monday
print("date            :", d0.date(), d0.day_name())
print("+ 3 business days:", (d0 + BDay(3)).date())
print("end of month     :", (d0 + MonthEnd(0)).date())
print("+ 1 month        :", (d0 + DateOffset(months=1)).date())
print("+ 45 days        :", (d0 + pd.Timedelta(days=45)).date())
print()
print("business days in Sep 2026:", len(pd.bdate_range("2026-09-01", "2026-09-30")))

## 10. Exercises

**E1.** Monthly revenue from `orders`, with a month-over-month percentage change column.

**E2.** For `daily_metrics`, compute a 30-day rolling average of `visits` and flag days where `visits` exceeded that average by more than 20%.

**E3.** Which day of the week has the highest average order amount?

**E4.** For each channel, the date of its best revenue month.

**E5.** Compute the number of days between each customer's signup and their first order.

In [ ]:
# Solutions
e1 = orders.resample("ME", on="order_ts")["amount"].sum().to_frame("revenue")
e1["mom_pct"] = (e1["revenue"].pct_change() * 100).round(1)
print("E1:\n", e1.head(5).round(0), "\n")

e2 = d[["visits"]].ffill().copy()
e2["ma30"] = e2["visits"].rolling(30, min_periods=1).mean()
e2["spike"] = e2["visits"] > e2["ma30"] * 1.2
print("E2: spike days =", int(e2["spike"].sum()), "of", len(e2), "\n")

e3 = (orders.assign(dow=lambda x: x["order_ts"].dt.day_name())
      .groupby("dow")["amount"].mean().round(2).sort_values(ascending=False))
print("E3:\n", e3, "\n")

m = (orders.set_index("order_ts").groupby("channel").resample("ME")["amount"].sum()
     .rename("revenue").reset_index())
e4 = m.loc[m.groupby("channel")["revenue"].idxmax()]
print("E4:\n", e4.round({"revenue": 0}).to_string(index=False), "\n")

first_order = orders.groupby("customer_id")["order_ts"].min().rename("first_order")
e5 = customers.merge(first_order, on="customer_id", how="left")
e5["days_to_first_order"] = (e5["first_order"] - e5["signup_date"]).dt.days
print("E5:\n", e5[["name", "signup_date", "first_order", "days_to_first_order"]].head(5).to_string(index=False))

## Recap

| SQL `OVER` clause | Pandas |
|---|---|
| `LAG(x, n)` | `s.shift(n)` |
| `LEAD(x, n)` | `s.shift(-n)` |
| `x - LAG(x)` | `s.diff()` |
| `SUM(x) OVER (ORDER BY t)` | `s.expanding().sum()` or `s.cumsum()` |
| `AVG(x) OVER (ORDER BY t ROWS 6 PRECEDING)` | `s.rolling(7).mean()` |
| `... PARTITION BY k` | `df.groupby("k")[col].transform(...)` |
| `DATE_TRUNC('month', ts)` + `GROUP BY` | `resample("ME")` |
| no equivalent | `ewm()`, `merge_asof()` |

- Put time in the index to unlock partial-string slicing and `resample`.
- `rolling(7)` = 7 rows; `rolling("7D")` = 7 calendar days. With irregular data you almost always want the second.
- Store in UTC, convert for display.
- Pandas 3.x frequency aliases: `ME`, `QE`, `YE`, `h`, `min`, `s`.

**Next:** `07_performance_and_gotchas.ipynb`.